<p align="center">
    <img src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Vertical-SinFondo.png" width="500"/>
</p>

<h2 align="center"><i>ITESO, Universidad Jesuita de Guadalajara</i></h2>
<h2 align="center"><i>Quantitative Finance</i></h2>
<h2 align="center"><i>Prof. Luis Carlos Alvarado Garnica</i></h2>

# Heston — Calibración de Mercado II 
---
En Calibración I montamos el pipeline. Aqui nos enfocamos en lo que hace la calibración **real** más dificil que la de libro de texto: los chequeos de no-arbitraje, el problema de identificabilidad, la validación, y los límites estructurales del modelo.

**Pipeline completo:**
1. Reunir quotes → 2. Limpiar datos → 3. Convertir a vol implicita → 4. Valor inicial → 5. Optimización global → 6. Refinamiento local → 7. **Validar y checar estabilidad**

## 0. Librerias y motor de pricing

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.stats import qmc
from scipy.integrate import quad
from scipy.optimize import differential_evolution, least_squares
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f9f9f9',
                     'axes.grid':True,'grid.alpha':0.35,'font.size':11,'lines.linewidth':2})
PURPLE='#534AB7'; GREEN='#1D9E75'; ORANGE='#D85A30'; AMBER='#BA7517'; BLUE='#2A7FBF'

def heston_cf_j(u,j,x,v,tau,r,kappa,theta,xi,rho):
    i=1j; uj=0.5 if j==1 else -0.5; bj=kappa-rho*xi if j==1 else kappa
    dj=np.sqrt((rho*xi*i*u-bj)**2+xi**2*(u**2-2*uj*i*u))
    g2j=(bj-rho*xi*i*u+dj)/(bj-rho*xi*i*u-dj)
    Dj=((bj-rho*xi*i*u+dj)/xi**2)*((1-np.exp(dj*tau))/(1-g2j*np.exp(dj*tau)))
    Cj=r*i*u*tau+(kappa*theta/xi**2)*((bj-rho*xi*i*u+dj)*tau-2*np.log((1-g2j*np.exp(dj*tau))/(1-g2j)))
    return np.exp(Cj+Dj*v+i*u*x)

def heston_prob_j(j,x,v,tau,K,r,kappa,theta,xi,rho):
    lnK=np.log(K)
    ig=lambda u:np.real(np.exp(-1j*u*lnK)*heston_cf_j(u,j,x,v,tau,r,kappa,theta,xi,rho)/(1j*u))
    val,_=quad(ig,1e-8,200,limit=200); return 0.5+val/np.pi

def heston_call(S,K,r,tau,v0,kappa,theta,xi,rho):
    x=np.log(S)
    return (S*heston_prob_j(1,x,v0,tau,K,r,kappa,theta,xi,rho)
            - K*np.exp(-r*tau)*heston_prob_j(2,x,v0,tau,K,r,kappa,theta,xi,rho))

def bs_call(S,K,r,tau,sigma):
    d1=(np.log(S/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau)); d2=d1-sigma*np.sqrt(tau)
    return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d2)

def bs_implied_vol(price,S,K,r,tau,tol=1e-7):
    intr=max(S-K*np.exp(-r*tau),0.0)
    if price<=intr+1e-8: return np.nan
    lo,hi=1e-6,5.0
    for _ in range(100):
        mid=(lo+hi)/2; d=bs_call(S,K,r,tau,mid)-price
        if abs(d)<tol: return mid
        lo,hi=(mid,hi) if d<0 else (lo,mid)
    return mid


## 1. Calibrar primero a una superficie sintética

Antes de confiar en la calibración con datos reales, la probamos donde **conocemos la respuesta**. El procedimiento:
1. Elegir parámetros "verdaderos" $\Theta_{\text{true}}$.
2. Generar precios del modelo en una malla (estos son "el mercado").
3. Correr el calibrador desde un punto de partida **distinto**.
4. Verificar que recupera $\Theta_{\text{true}}$.

Si la recuperación falla en datos sintéticos limpios, el bug está en el calibrador, no en el mercado.


In [ ]:
# Código aquí

## 2. Chequeos de no-arbitraje

Una superficie con arbitraje **no puede** ser ajustada por ningún modelo libre de arbitraje (Heston lo es). Antes de calibrar verificamos las condiciones estáticas:

- **Monotonia:** los precios de call son no crecientes en el strike.
- **Convexidad:** los precios de call son convexos en el strike (mariposa $\geq 0$).
- **Calendario:** opciones de mayor plazo valen al menos lo mismo que las de menor plazo (mismo strike).

Si saltamos estos chequeos, el optimizador "pelea" contra el arbitraje y aterriza en parámetros distorsionados.


In [ ]:
def chequeos_arbitraje(market, S0):
    pass

# rep = chequeos_arbitraje(market, S0)
# print("Chequeos de no-arbitraje por vencimiento:")
# for T, res in rep.items():
#     estado = "OK" if (res['monotonia'] and res['convexidad']) else "FALLA"
#     print(f"  T={T:.2f}a  mono={res['monotonia']}  conv={res['convexidad']}  "
#           f"min_butterfly={res['min_butterfly']:.4f}  [{estado}]")


## 3. Correr la calibración desde otro punto de partida

Arrancamos deliberadamente desde valores distintos a los verdaderos, para ver si el calibrador los encuentra.


In [ ]:
def loss(params, market, S0, r):
    v0, theta, kappa, xi, rho = params
    total = 0.0
    for K, T, price_mkt, spread in market:
        try:
            price_mdl = heston_call(S0, K, r, T, v0, kappa, theta, xi, rho)
            w = 1.0 / max(spread, 0.01)**2          # peso por liquidez
            total += w * (price_mdl - price_mkt)**2
        except Exception:
            total += 1e6                             # penaliza params invalidos
    return total

bounds=[(0.005,0.20),(0.005,0.20),(0.10,6.0),(0.05,1.5),(-0.95,0.5)]

print("Etapa 1+2: busqueda global rapida (Latin Hypercube) + refinamiento local (least_squares/TRF)...")

def residuals(params):
    v0,theta,kappa,xi,rho = params
    res=[]

    for K,T,price_mkt,spread in market:
        w = 1.0/max(spread,0.01)
        try:
            pm = heston_call(S0,K,r,T,v0,kappa,theta,xi,rho)
            res.append(w*(pm - price_mkt))
        except Exception:
            res.append(1e3)
    return res

def busqueda_global_rapida(bounds, loss, market, S0, r, n_candidatos=40, n_refinar=4, seed=1):
    lb = np.array([b[0] for b in bounds]); ub = np.array([b[1] for b in bounds])
    muestra = qmc.LatinHypercube(d=len(bounds), seed=seed).random(n_candidatos)
    candidatos = lb + muestra*(ub-lb)
    perdidas = [loss(c, market, S0, r) for c in candidatos]
    mejores_idx = np.argsort(perdidas)[:n_refinar]
    mejor_fit = None
    i = 1
    print(f"Refinando localmente los {n_refinar} mejores candidatos del escaneo global...")
    for idx in mejores_idx:
        fit = least_squares(residuals, candidatos[idx], bounds=(lb,ub), max_nfev=100)
        print(f"Fit {i} de {n_refinar}: costo={2*fit.cost:.4f}, params={fit.x}")
        i += 1
        if mejor_fit is None or fit.cost < mejor_fit.cost:
            mejor_fit = fit
    return mejor_fit



In [ ]:
# Código de comparación aquí

## 4. El problema de identificabilidad $\kappa$–$\xi$

Observa el resultado anterior: es muy probable que $v_0$ y $\rho$ se recuperen casi perfectos, pero $\kappa$, $\xi$ y $\theta$ no exactamente, **aunque el RMSE en precio sea diminuto**.

La causa: la superficie de pérdida tiene **valles planos**. Muchas combinaciones $(\kappa,\xi)$ precian la superficie casi idénticamente. Los datos no los fijan. Vamos a *ver* ese valle.


In [ ]:
# Barremos kappa y xi alrededor del optimo, reoptimizando el resto seria caro;
# aqui fijamos v0,theta,rho en los verdaderos y vemos el valle en (kappa,xi).

# Código aquí


## 5. Mitigaciones del problema de identificabilidad

Hay tres remedios prácticos:

- **Regularizar:** añadir una penalización que jale los parámetros hacia los de ayer (estabilidad temporal).
- **Fijar $\kappa$:** algunos escritorios fijan $\kappa$ a una constante sensata y calibran los otros cuatro.
- **Añadir más vencimientos:** la estructura temporal ayuda a separar $\kappa$ de $\xi$.

Demostramos la segunda: fijar $\kappa$ y calibrar cuatro parámetros da un ajuste igual de bueno con parámetros más estables.


In [ ]:
# Código aquí


## 6. Validación del ajuste

Métricas cuantitativas y chequeos cualitativos:
- **RMSE** en puntos de vol implicita.
- **Error máximo** en la superficie.
- **Sensatez económica** de los parámetros ($\rho<0$ para equities, $\kappa$ no absurdo).


In [ ]:
# fig, ax = plt.subplots(figsize=(11,6))
# errores=[]
# for T in sorted(set(row[1] for row in market)):
#     filas=[row for row in market if row[1]==T]
#     Ks=np.array([row[0] for row in filas]); orden=np.argsort(Ks); Ks=Ks[orden]
#     iv_mkt=np.array([bs_implied_vol(row[2],S0,row[0],r,T)*100 for row in filas])[orden]
#     iv_mdl=[]
#     for K in Ks:
#         pm=heston_call(S0,K,r,T,mejor_fit_k.x[0],mejor_fit_k.x[2],mejor_fit_k.x[1],mejor_fit_k.x[3],mejor_fit_k.x[4])
#         iv_mdl.append(bs_implied_vol(pm,S0,K,r,T)*100)
#     iv_mdl=np.array(iv_mdl); errores.extend(iv_mdl-iv_mkt)
#     l=ax.plot(Ks,iv_mkt,'o',label=f'Mkt T={T:.2f}')[0]
#     ax.plot(Ks,iv_mdl,'-',color=l.get_color(),label=f'Heston T={T:.2f}')
# ax.axvline(S0,color='gray',ls=':')
# ax.set_xlabel('Strike K'); ax.set_ylabel('Vol implicita (%)')
# ax.set_title('Validacion: modelo vs mercado por vencimiento'); ax.legend(fontsize=8,ncol=2)
# plt.tight_layout(); plt.show()

# errores=np.array(errores)
# print(f"RMSE:        {np.sqrt(np.mean(errores**2)):.4f} puntos de vol")
# print(f"Error maximo:{np.max(np.abs(errores)):.4f} puntos de vol")
# print(f"rho<0 (equity-sensible): {'SI' if mejor_fit_k.x[4]<0 else 'NO'}")


## 7. Los límites estructurales de Heston

Aunque la calibración sea perfecta, el modelo tiene limites que ningún ajuste esconde:

- **Skew de corto plazo:** Heston no genera suficiente skew para vencimientos muy cortos (semanas). La difusión continua no basta.
- **Sin saltos:** los mercados reales dan gaps. Los caminos continuos de Heston subvaluan las alas de corto plazo. Extensiones añaden saltos (modelo de **Bates**).
- **Un solo factor:** una sola varianza no captura toda la estructura temporal del skew. Modelos multi-factor o de **volatilidad rugosa** lo abordan.

**Dónde Heston sigue ganando:** vencimientos de mediano a largo plazo, y como benchmark transparente y rápido de calibrar.
